# FinGuard Kredi Risk Analizi ve Veri Ön İşleme

Bu Jupyter Notebook, SentinelBank API projesi kapsamındaki kredi risk analizinde kullanılacak olan makine öğrenmesi veri setini yüklemek, analiz etmek ve temizlemek amacıyla hazırlanmıştır.

### Adım 2: Gerekli Kütüphanelerin Yüklenmesi ve Verinin Yüklenmesi

Bu adımda `pandas` ve `numpy` kütüphanelerini import edip, veriyi `pd.read_csv()` ile yükleyeceğiz.

In [ ]:
import pandas as pd
import numpy as np

# Kolonların tamamını rahatça görebilmek için pandas gösterim ayarını yapılandıralım
pd.set_option('display.max_columns', None)

In [ ]:
# Veriyi yükleyelim
df = pd.read_csv("credit_risk_dataset.csv")

### Adım 3: Veri İnceleme (Exploratory Data Analysis - EDA)

Verideki eksik elemanlar, kolon tipleri ve veri dağılımlarını `df.head()`, `df.info()` ve `df.isnull().sum()` komutlarıyla inceleyelim.

In [ ]:
# İlk 5 satırı görüntüleyelim
df.head()

   person_age  person_income person_home_ownership  person_emp_length loan_intent loan_grade  loan_amnt  loan_int_rate  loan_status  loan_percent_income cb_person_default_on_file  cb_person_cred_hist_length
0          22          59000                  RENT              123.0    PERSONAL          D      35000          16.02            1                 0.59                         Y                           3
1          21           9600                   OWN                5.0   EDUCATION          B       1000          11.14            0                 0.10                         N                           2
2          25           9600              MORTGAGE                1.0     MEDICAL          C       5500          12.87            1                 0.57                         N                           3
3          23          65500                  RENT                4.0     MEDICAL          C      35000          15.23            1                 0.53                    

In [ ]:
# Veri setinin yapısal bilgileri, veri türleri ve null olmayan satır sayıları
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  str    
 3   person_emp_length           31686 non-null  float64
 4   loan_intent                 32581 non-null  str    
 5   loan_grade                  32581 non-null  str    
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               29465 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  str    
 11  cb_person_cred_hist_length  32581 non-null  int64  
dtypes: float64(3), int64(5), str(4)
memory usage: 3.0 MB


In [ ]:
# Hangi kolonda kaç adet eksik (null) değer olduğunu sorgulayalım
df.isnull().sum()

person_age                       0
person_income                    0
person_home_ownership            0
person_emp_length              895
loan_intent                      0
loan_grade                       0
loan_amnt                        0
loan_int_rate                 3116
loan_status                      0
loan_percent_income              0
cb_person_default_on_file        0
cb_person_cred_hist_length       0


### Veri Temizleme ve Eksik Verileri Doldurma

**Tespit Edilen Durumlar:**
1. `person_age` (Yaş) kolonunda 100'ün üzerinde mantıksız yaşlar (123, 144) bulunmaktadır.
2. `person_emp_length` (Çalışma Süresi) kolonunda hem **895 adet eksik değer** vardır, hem de 123 yıl gibi mantıksız bir uç değer bulunmaktadır.
3. `loan_int_rate` (Kredi Faiz Oranı) kolonunda **3116 adet eksik değer** bulunmaktadır.

**Temizleme Stratejimiz:**
- **Uç Değerlerin Temizlenmesi:**
  - Yaşı 100'den büyük olan satırları filtreleyeceğiz.
  - Çalışma süresi 60 yıldan büyük olan ya da kişinin yaşından büyük olan mantık dışı satırları filtreleyeceğiz.
- **Eksik Verilerin Doldurulması:**
  - Çalışma süresi (`person_emp_length`) eksik değerlerini genel **medyan** değeriyle dolduracağız.
  - Kredi faiz oranları (`loan_int_rate`) doğrudan kredi notuna (`loan_grade`) bağlı olduğu için, her kredi notu grubunun kendi **medyan** faiz oranı hesaplanarak eksik değerler buna göre doldurulacaktır.

In [ ]:
print(f"Temizlik öncesi veri boyutu: {df.shape}")

# 1. Yaş uç değerlerini temizleyelim (yaş > 100 olanlar)
df = df[df['person_age'] <= 100]

# 2. Çalışma süresi uç değerlerini temizleyelim (null olanlar hariç tutularak mantıksız olanlar silinir)
df = df[df['person_emp_length'].isnull() | ((df['person_emp_length'] <= 60) & (df['person_emp_length'] < df['person_age']))]

print(f"Uç değerler silindikten sonra veri boyutu: {df.shape}")

Temizlik öncesi veri boyutu: (32581, 12)
Uç değerler silindikten sonra veri boyutu: (32574, 12)


In [ ]:
# 3. Çalışma süresi eksik değerlerini medyan ile dolduralım
emp_length_median = df['person_emp_length'].median()
print(f"Çalışma süresi medyanı: {emp_length_median}")
df['person_emp_length'] = df['person_emp_length'].fillna(emp_length_median)

# 4. Kredi faiz oranı eksik değerlerini loan_grade bazlı medyanlar ile dolduralım
grade_medians = df.groupby('loan_grade')['loan_int_rate'].transform('median')
df['loan_int_rate'] = df['loan_int_rate'].fillna(grade_medians)

print("Eksik değerler başarıyla dolduruldu.")

Çalışma süresi medyanı: 4.0
Eksik değerler başarıyla dolduruldu.


In [ ]:
# Temizlik ve doldurma işlemlerinin doğruluğunu kontrol edelim
print("--- Kalan Eksik Değer Sayıları ---")
print(df.isnull().sum())

print("\n--- Veri Özeti (Describe) ---")
df.describe()

--- Kalan Eksik Değer Sayıları ---
person_age                    0
person_income                 0
person_home_ownership         0
person_emp_length             0
loan_intent                   0
loan_grade                    0
loan_amnt                     0
loan_int_rate                 0
loan_status                   0
loan_percent_income           0
cb_person_default_on_file     0
cb_person_cred_hist_length    0
dtype: int64

--- Veri Özeti (Describe) ---


         person_age  person_income  person_emp_length     loan_amnt  loan_int_rate   loan_status  loan_percent_income  cb_person_cred_hist_length
count  32574.000000   3.257400e+04       32574.000000  32574.000000   32574.000000  32574.000000         32574.000000                32574.000000
mean      27.718426   6.587848e+04           4.760576   9588.018051      11.013752      0.218180             0.170202                    5.804108
std        6.204987   5.253194e+04           3.981181   6320.249598       3.212328      0.413017             0.106755                    4.053873
min       20.000000   4.000000e+03           0.000000    500.000000       5.420000      0.000000             0.000000                    2.000000
25%       23.000000   3.850000e+04           2.000000   5000.000000       7.880000      0.000000             0.090000                    3.000000
50%       26.000000   5.500000e+04           4.000000   8000.000000      10.990000      0.000000             0.150000       

### Adım 4: Veri Ön İşleme (Encoding, Split ve Scaling)

Bu adımda aşağıdaki işlemleri gerçekleştireceğiz:
1. **Kategorik verilerin sayısal verilere dönüştürülmesi**: `pd.get_dummies()` kullanarak One-Hot Encoding uygulayacağız.
2. **Özellikler (X) ve Hedef Değişkenin (y) Ayrılması**: Hedef değişkenimiz olan `loan_status` (kredi onaylandı mı: 1, reddedildi mi: 0) ayrılacaktır.
3. **Eğitim ve Test Setlerinin Oluşturulması**: Verinin %80'i eğitim, %20'si test seti olarak `train_test_split` ile ayrılacaktır.
4. **Ölçeklendirme (Feature Scaling)**: Sayısal değerlerin ölçeklerini eşitlemek için `StandardScaler` uygulayacağız.

In [ ]:
# 1. Kategorik kolonları belirleyelim
categorical_cols = ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']

# pd.get_dummies kullanarak One-Hot Encoding uygulayalım (kukla değişken tuzağını önlemek için drop_first=True yapıyoruz)
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("One-Hot Encoding sonrası veri seti kolonları:")
df_encoded.head()

In [ ]:
# 2. Veriyi özellikler (X) ve hedef değişken (y) olarak ikiye bölelim
X = df_encoded.drop(columns=['loan_status'])
y = df_encoded['loan_status']

print(f"X boyutu: {X.shape}, y boyutu: {y.shape}")

In [ ]:
from sklearn.model_selection import train_test_split

# 3. train_test_split ile verinin %80'ini train, %20'sini test olarak ayıralım
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Eğitim seti (X_train) boyutu: {X_train.shape}")
print(f"Test seti (X_test) boyutu: {X_test.shape}")

In [ ]:
from sklearn.preprocessing import StandardScaler

# Ölçeklendirilecek sayısal kolonlar (One-Hot Encoding ile üretilen 0/1 kolonlar hariç)
numeric_cols = ['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length']

# 4. StandardScaler uygulayalım
scaler = StandardScaler()

# Eğitim verisi üzerinde fit_transform, test verisi üzerinde transform uyguluyoruz
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

# Ölçeklendirilmiş verilerin ilk 5 satırını inceleyelim
X_train_scaled.head()

### Adım 5: Model Eğitimi, Değerlendirme ve Shannon Entropy ile Belirsizlik Analizi

Bu adımda aşağıdaki işlemleri gerçekleştireceğiz:
1. **Random Forest Sınıflandırıcı Modeli Eğitimi**: Eğitim verileri (`X_train_scaled`, `y_train`) kullanılarak model eğitilecektir.
2. **Model Performansının Değerlendirilmesi**: Test verisi üzerinde tahminler yapıp `classification_report` ve `confusion_matrix` çıktılarını inceleyeceğiz.
3. **Shannon Entropy ile Belirsizlik Skoru (Uncertainty Score) Hesaplama**: Modelin tahmin olasılıklarını kullanarak Shannon Entropy hesaplayacağız. Entropy değeri yüksek olan (belirsiz) tahminleri tespit ederek manuel incelemeye yönlendireceğiz.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 1. Random Forest modelini tanımlayıp eğitelim
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

print("Random Forest modeli başarıyla eğitildi!")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# 2. Test setinde tahminler yapalım
y_pred = rf_model.predict(X_test_scaled)

# Hata Matrisi ve Sınıflandırma Raporu çıktıları
print("--- Confusion Matrix (Hata Matrisi) ---")
print(confusion_matrix(y_test, y_pred))

print("\n--- Classification Report (Sınıflandırma Raporu) ---")
print(classification_report(y_test, y_pred))

In [ ]:
# 3. Modelin tahmin olasılıklarını (predict_proba) alalım
y_proba = rf_model.predict_proba(X_test_scaled)

# Shannon Entropy hesaplama fonksiyonu
def calculate_entropy(probs):
    # Sıfır değerinde log hatasını (log(0) tanımsızdır) önlemek için clipping (sınırlandırma) uyguluyoruz
    eps = 1e-15
    probs = np.clip(probs, eps, 1 - eps)
    # Shannon Entropy Formülü: H(p) = -sum(p * log2(p))
    return -np.sum(probs * np.log2(probs), axis=1)

# Her bir tahmin için belirsizlik skorunu hesaplayalım
entropies = calculate_entropy(y_proba)

# Sonuçları incelemek için bir veri çerçevesi (DataFrame) oluşturalım
results_df = pd.DataFrame({
    'Gerçek_Değer': y_test,
    'Tahmin_Edilen': y_pred,
    'Kredi_Onay_Olasılığı_0': y_proba[:, 0],
    'Kredi_Ret_Olasılığı_1': y_proba[:, 1],
    'Belirsizlik_Skoru_Entropy': entropies
})

# Entropy > 0.8 olan (tahmini belirsiz, örn: 55/45 veya 60/40 gibi dağılmış) durumları belirleyelim
# İki sınıflı problemde maksimum entropy 1.0'dır (yarı yarıya olasılık durumu)
threshold = 0.8
results_df['Manuel_İnceleme_Gerekli'] = results_df['Belirsizlik_Skoru_Entropy'] > threshold

print(f"Toplam test veri sayısı: {len(results_df)}")
uncertain_cases = results_df[results_df['Manuel_İnceleme_Gerekli']]
print(f"Manuel incelemeye gönderilecek belirsiz veri sayısı (Entropy > {threshold}): {len(uncertain_cases)}")
print(f"Oran: %{len(uncertain_cases) / len(results_df) * 100:.2f}")

print("\nEn Belirsiz 5 Örnek (Olasılıkları ve Entropy değerleri):")
results_df[results_df['Manuel_İnceleme_Gerekli']].head()